# Cosponsorship-network features and tilt
The cosponsorship graph (`cosponsorship/data/`, built by `build_cosponsor.py`) links two House members when they take part in the same bill (sponsor or non-withdrawn cosponsor), over *all* House bills and resolutions.

We rebuild the member--member projection here from `sponsorship.csv`, using **only cosponsorships dated before 1 November of the election year** (no lame-duck leakage), under three edge weightings:
- `raw`: number of shared bills;
- `norm`: sum of 1/(k-1) over shared bills (k = participants), which very aggressively favours tiny bills;
- `idf`: sum of log(N/k) over shared bills, a smoother discount of bills nearly everyone signs.

Question: **does the weighting change what the cosponsorship graph tells us about tilt?** Setup as before: train 115/116, test 118, 117 set aside; targets `tilt` and `tilt_c`.

In [1]:
import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
TRAIN, TEST = [115, 116], 118
CONGS = TRAIN + [TEST]
SCHEMES = ["raw", "norm", "idf"]
K_NN = 15

## 1. Features from the cosponsorship graph

In [2]:
def cos_feats(c):
    ey = 2018 + 2 * (c - 115)
    sp = pd.read_csv(f"data/{c}/sponsorship.csv")
    sp = sp[(~sp.withdrawn) & (sp.date.fillna("0000") <= f"{ey}-11-01")]
    mem = pd.read_csv(f"data/{c}/members.csv").set_index("bioguide")
    members = sorted(sp.member.unique()); mi = {m: i for i, m in enumerate(members)}
    bills = sp.bill.unique(); bi = {b: j for j, b in enumerate(bills)}
    B = np.zeros((len(members), len(bills)), dtype=np.float32); S = np.zeros_like(B)
    for r in sp.itertuples():
        i, j = mi[r.member], bi[r.bill]; B[i, j] = 1
        if r.role == "sponsor": S[i, j] = 1
    C = B - S
    k = B.sum(0); N = len(members)
    W = {"raw": B @ B.T, "norm": (B / np.maximum(k - 1, 1)) @ B.T, "idf": (B * np.log(N / np.maximum(k, 1))) @ B.T}
    party = mem.party.reindex(members).to_numpy(dtype=object)
    other = (party[:, None] != party[None, :])

    f = pd.DataFrame(index=members)
    # ---- weight-independent behaviour
    f["n_sponsored"] = np.log1p(S.sum(1)); f["n_cosponsored"] = np.log1p(C.sum(1))
    f["mean_k_cosponsored"] = (C @ k) / np.maximum(C.sum(1), 1)                    # do they sign mass resolutions or niche bills?
    f["niche_cosponsor_share"] = (C * (k <= 20)).sum(1) / np.maximum(C.sum(1), 1)
    spon_party = sp[sp.role == "sponsor"].drop_duplicates("bill").set_index("bill").member.map(mem.party)
    cs = sp[sp.role == "cosponsor"].copy()
    cs["sp_party"], cs["m_party"] = cs.bill.map(spon_party), cs.member.map(mem.party)
    cs["cross"] = (cs.sp_party != cs.m_party) & cs.sp_party.notna()
    f["cross_cosponsor_rate"] = cs.groupby("member").cross.mean().reindex(members)  # share of their cosponsorships on the other party's bills
    cs["bill_other"] = (cs.m_party != cs.bill.map(spon_party))                       # cosponsor from the other party than the bill's sponsor
    drawn = cs[cs.bill_other].groupby("bill").size()
    own = sp[sp.role == "sponsor"].set_index("member")
    f["bipartisan_draw_rate"] = own.bill.map(lambda b: b in drawn.index).groupby(level=0).mean().reindex(members)  # share of own bills drawing an other-party cosponsor
    # ---- weighted network features, one block per weighting
    for s in SCHEMES:
        M = W[s].copy(); np.fill_diagonal(M, 0)
        strength = M.sum(1)
        f[f"{s}_strength"] = np.log1p(strength)
        f[f"{s}_cross_share"] = (M * other).sum(1) / np.maximum(strength, 1e-9)
        f[f"{s}_eigcent"] = np.abs(np.linalg.eigh(M)[1][:, -1])
        f[f"{s}_pagerank"] = pd.Series(nx.pagerank(nx.from_numpy_array(M), weight="weight")).values
        nb = np.argsort(-M, axis=1)[:, :K_NN]
        K = nx.Graph(); K.add_nodes_from(range(N)); K.add_edges_from((i, j) for i in range(N) for j in nb[i])
        f[f"{s}_betweenness"] = pd.Series(nx.betweenness_centrality(K)).values
        f[f"{s}_knn_cross"] = np.array([other[i, nb[i]].mean() for i in range(N)])
    return f.assign(congress=c).rename_axis("bioguide").reset_index()

cos = pd.concat([cos_feats(c) for c in CONGS], ignore_index=True)
IND = ["n_sponsored", "n_cosponsored", "mean_k_cosponsored", "niche_cosponsor_share", "cross_cosponsor_rate", "bipartisan_draw_rate"]
NET = {s: [c for c in cos.columns if c.startswith(s + "_")] for s in SCHEMES}
print(cos.groupby("congress").size().to_dict(), "members;", len(IND), "weight-independent +", len(NET["idf"]), "per-weighting features")
cos[IND + NET["idf"]].describe().T[["mean", "std", "min", "max"]].round(3)

{115: 448, 116: 447, 118: 449} members; 6 weight-independent + 6 per-weighting features


,mean,std,min,max
n_sponsored,2.960,0.684,0.000,6.409
n_cosponsored,5.668,0.682,1.099,7.618
mean_k_cosponsored,90.615,20.058,15.000,174.923
niche_cosponsor_share,0.227,0.087,0.000,0.855
cross_cosponsor_rate,0.215,0.150,0.000,1.000
bipartisan_draw_rate,0.449,0.231,0.000,1.000
idf_strength,10.312,0.701,5.117,12.044
idf_cross_share,0.229,0.129,0.029,1.000
idf_eigcent,0.037,0.030,0.000,0.191
idf_pagerank,0.002,0.001,0.000,0.008


## 2. Disloyalty (from votes, as before) and the target

In [3]:
def disloyalty(c):
    mem = pd.read_csv(f"../data/{c}/voteview/H{c}_members.csv")
    mem = mem[(mem.chamber == "House") & mem.bioguide_id.notna()].drop_duplicates("bioguide_id")
    bills = pd.read_csv(f"../data/{c}/bills.csv")[["bill_number", "rollnumber"]]
    votes = pd.read_csv(f"../data/{c}/voteview/H{c}_votes.csv")
    votes = votes[votes.chamber == "House"].merge(bills, on="rollnumber")
    votes["v"] = votes.cast_code.map({1: 1, 2: 1, 3: 1, 4: -1, 5: -1, 6: -1}).fillna(0)
    v = votes.pivot_table(index="icpsr", columns="bill_number", values="v", aggfunc="first").reindex(index=mem.icpsr, columns=bills.bill_number).fillna(0).values
    p = mem.party_code.map({100: "D", 200: "R"}).fillna("O").values
    n_cross, n_line = np.zeros(len(v)), np.zeros(len(v))
    for q in ["D", "R"]:
        m = p == q; tot, pres = v[m].sum(0), np.abs(v[m]).sum(0)
        line = (pres > 0) & (np.abs(tot) / np.maximum(pres, 1) >= 0.5); maj = np.sign(tot)
        n_cross[m] = ((v[m] != 0) & (v[m] != maj) & line).sum(1); n_line[m] = ((v[m] != 0) & line).sum(1)
    return pd.DataFrame({"bioguide": mem.bioguide_id.values, "n_cross": np.log1p(n_cross), "cross_rate": n_cross / np.maximum(n_line, 1)}).assign(congress=c)

dis = pd.concat([disloyalty(c) for c in CONGS])
G_DIS = ["n_cross", "cross_rate"]
targets = pd.read_csv("../data/elections/targets.csv")
d = targets.merge(cos, on=["congress", "bioguide"], how="left").merge(dis, on=["congress", "bioguide"], how="left")
d = d[d.tilt.notna() & d.party.isin(["D", "R"]) & d.congress.isin(CONGS)].copy()
d["tilt_c"] = d.tilt - d.groupby(["congress", "party"]).tilt.transform("mean")
d["is_R"] = (d.party == "R").astype(float); d["prev_share"] = d.prev_two_party_share
ALLF = IND + sum(NET.values(), []) + G_DIS
z = lambda x: (x - x.mean()) / x.std()
print("members with a cosponsorship record:", d.groupby("congress").n_sponsored.apply(lambda s: s.notna().mean()).round(3).to_dict())
d[ALLF + ["prev_share"]] = d.groupby("congress")[ALLF + ["prev_share"]].transform(z)
d[ALLF] = d[ALLF].fillna(0)
print(d.groupby("congress").size().to_dict(), "rows")

members with a cosponsorship record: {115: 1.0, 116: 1.0, 118: 1.0}
{115: 281, 116: 327, 118: 320} rows


## 3. Univariate: Spearman with `tilt_c` by Congress, and sign stability

In [4]:
rows = []
for f_ in IND + [c for s in SCHEMES for c in NET[s]] + G_DIS + ["prev_share"]:
    r = {"feature": f_}
    for c in CONGS:
        s = d[d.congress == c]; r[c] = spearmanr(s[f_], s.tilt_c)[0]
    rows.append(r)
uni = pd.DataFrame(rows).set_index("feature")
uni["same_sign"] = (np.sign(uni[115]) == np.sign(uni[116])) & (np.sign(uni[116]) == np.sign(uni[118]))
uni["mean_abs"] = uni[CONGS].abs().mean(axis=1)
uni.round(3)

,115,116,118,same_sign,mean_abs
feature,,,,,
n_sponsored,-0.157,-0.043,0.030,False,0.077
n_cosponsored,-0.078,0.049,0.057,False,0.062
mean_k_cosponsored,0.078,0.157,-0.023,False,0.086
niche_cosponsor_share,0.000,-0.068,0.033,False,0.034
cross_cosponsor_rate,-0.142,0.019,0.046,False,0.069
bipartisan_draw_rate,-0.015,0.081,-0.018,False,0.038
raw_strength,-0.071,0.099,0.057,False,0.075
raw_cross_share,-0.100,0.013,0.044,False,0.052
raw_eigcent,-0.080,0.099,0.038,False,0.072


## 4. Models: how much does the weighting matter?
Ridge (penalty by leave-one-Congress-out on 115/116), scored on 118; `oos R2` vs the training mean. Pooled model with `is_R`, and separate per party.

In [5]:
def ridge(X, y, lam):
    mu, ym = X.mean(0), y.mean()
    beta = np.linalg.solve((X - mu).T @ (X - mu) + lam * np.eye(X.shape[1]), (X - mu).T @ (y - ym))
    return (lambda Xn: ym + (Xn - mu) @ beta), beta

def tune_fit(tr_, cols, target):
    cv = {lam: sum(((b[target].values - ridge(a[cols].values, a[target].values, lam)[0](b[cols].values)) ** 2).sum()
                   for a, b in [(tr_[tr_.congress != h], tr_[tr_.congress == h]) for h in TRAIN]) for lam in [1, 10, 100, 1000, 10000]}
    lam = min(cv, key=cv.get); return ridge(tr_[cols].values, tr_[target].values, lam), lam

tr_all, te = d[d.congress.isin(TRAIN)], d[d.congress == TEST]
def pooled(cols, target):
    (f, beta), lam = tune_fit(tr_all, cols, target); p, y = f(te[cols].values), te[target].values
    return 1 - ((y - p) ** 2).sum() / ((y - tr_all[target].mean()) ** 2).sum(), pearsonr(p, y)[0]

def by_party(cols, target):
    p = np.zeros(len(te)); mus = {}
    for q in ["D", "R"]:
        tr_q = tr_all[tr_all.party == q]; (f, _), _ = tune_fit(tr_q, cols, target)
        m = (te.party == q).values; p[m] = f(te[cols].values[m]); mus[q] = tr_q[target].mean()
    y = te[target].values
    r2 = {q: 1 - ((y[(te.party == q).values] - p[(te.party == q).values]) ** 2).sum() / ((y[(te.party == q).values] - mus[q]) ** 2).sum() for q in ["D", "R"]}
    return 1 - ((y - p) ** 2).sum() / ((y - tr_all[target].mean()) ** 2).sum(), r2["D"], r2["R"]

B0, B1 = ["prev_share"], ["prev_share"] + G_DIS
fsets = {"prev share": B0, "prev share + disloyalty": B1,
         "prev share + weight-free cosponsor behaviour": B0 + IND,
         **{f"prev share + network [{s}]": B0 + NET[s] for s in SCHEMES},
         **{f"prev share + behaviour + network [{s}]": B0 + IND + NET[s] for s in SCHEMES},
         "prev share + disloyalty + behaviour + network [idf]": B1 + IND + NET["idf"],
         "network [idf] only": NET["idf"], "behaviour only": IND}
rows = []
for target in ["tilt_c", "tilt"]:
    for name, cols in fsets.items():
        r2, corr = pooled(["is_R"] + cols, target); both, rd, rr = by_party(cols, target)
        rows.append({"target": target, "features": name, "pooled R2": r2, "pooled corr": corr, "separate R2": both, "D R2": rd, "R R2": rr})
res = pd.DataFrame(rows).set_index(["target", "features"]); res.round(3)

pooled R2  pooled corr  separate R2   D R2    R R2
target features                                                                                              
tilt_c prev share                                              0.077        0.286        0.083  0.111   0.024
       prev share + disloyalty                                 0.087        0.307        0.096  0.134   0.018
       prev share + weight-free cosponsor behaviour            0.074        0.279        0.079  0.117   0.001
       prev share + network [raw]                              0.083        0.299        0.108  0.159   0.001
       prev share + network [norm]                             0.079        0.285        0.091  0.135   0.000
       prev share + network [idf]                              0.087        0.306        0.107  0.158   0.000
       prev share + behaviour + network [raw]                  0.036        0.225        0.103  0.152   0.000
       prev share + behaviour + network [norm]                 0.073        0.269        0.084  0.124  -0.000
       prev share + behaviour + network [idf]                  0.037        0.233        0.105  0.155  -0.000
       prev share + disloyalty + behaviour + network [...      0.044        0.261        0.102  0.150  -0.000
       network [idf] only                                     -0.001       -0.040       -0.000 -0.000  -0.001
       behaviour only                                         -0.009       -0.030       -0.008 -0.011  -0.001
tilt   prev share                                              0.007        0.272        0.060  0.117   0.004
       prev share + disloyalty                                 0.006        0.199        0.051  0.107  -0.000
       prev share + weight-free cosponsor behaviour            0.006        0.116        0.056  0.115  -0.004
       prev share + network [raw]                              0.001        0.033        0.047  0.102  -0.001
       prev share + network [norm]                             0.004        0.099      -25.009  0.068 -82.692
       prev share + network [idf]                              0.001        0.044       -0.053  0.106  -0.341
       prev share + behaviour + network [raw]                  0.000        0.023        0.074  0.142  -0.006
       prev share + behaviour + network [norm]                 0.003        0.055       -2.345  0.029  -7.727
       prev share + behaviour + network [idf]                  0.001        0.028       -0.069  0.141  -0.478
       prev share + disloyalty + behaviour + network [...     -0.000        0.016       -0.082  0.117  -0.462
       network [idf] only                                     -0.006       -0.129       -0.027 -0.000  -0.002
       behaviour only                                         -0.002       -0.024       -0.028  0.000  -0.005

## 5. Same question, one feature at a time: does the weighting change each network feature's link to tilt?

In [6]:
rows = []
for kind in ["strength", "cross_share", "eigcent", "pagerank", "betweenness", "knn_cross"]:
    r = {"feature": kind}
    for s in SCHEMES:
        rho = [spearmanr(d[d.congress == c][f"{s}_{kind}"], d[d.congress == c].tilt_c)[0] for c in CONGS]
        r.update({f"{s} (115/116/118)": " / ".join(f"{x:+.2f}" for x in rho)})
    rows.append(r)
pd.DataFrame(rows).set_index("feature")

,raw (115/116/118),norm (115/116/118),idf (115/116/118)
feature,,,
strength,-0.07 / +0.10 / +0.06,-0.08 / +0.04 / +0.05,-0.07 / +0.07 / +0.06
cross_share,-0.10 / +0.01 / +0.04,-0.12 / +0.03 / +0.04,-0.11 / +0.02 / +0.05
eigcent,-0.08 / +0.10 / +0.04,-0.10 / +0.06 / +0.06,-0.09 / +0.08 / +0.03
pagerank,-0.07 / +0.10 / +0.06,-0.08 / +0.03 / +0.07,-0.07 / +0.06 / +0.07
betweenness,-0.04 / -0.04 / +0.15,-0.06 / -0.02 / +0.08,-0.04 / -0.04 / +0.13
knn_cross,-0.14 / +0.06 / +0.01,-0.11 / +0.06 / +0.07,-0.12 / +0.04 / +0.04


## What we notice
1. **The weighting barely matters for prediction.** The three edge weightings give almost identical per-feature relationships with `tilt_c` (section 5), and pooled out-of-sample R2 for prev share + network features is 0.083 (raw), 0.079 (1/(k-1)), 0.087 (IDF) against 0.077 for prev share alone. Differences of ~0.005 are noise with one test year.
2. **No cosponsorship feature has a stable sign.** Every network and behaviour feature is negative in 2018 and positive in 2020 / 2024 (section 3), so none is a reliable predictor. Only `prev_share` and disloyalty keep a sign across all three years.
3. **Where weighting does show up is robustness.** On raw `tilt` the 1/(k-1) weighting makes the per-party model blow up (R2 of -25 overall, -83 for Republicans): its features are heavy-tailed (tiny bills dominate), so a few extreme test members get wild predictions. IDF and raw count are stable.
4. **Democrats gain, Republicans do not.** Per-party models on `tilt_c`: Democrats 0.111 (prev share) -> 0.158 with raw or IDF network features; Republicans fall to ~0.00. Overall separate-model R2 0.083 -> 0.107/0.108. Suggestive only: ~160 Democrats in one test year.
5. **Stacking more features hurts.** Adding the six weight-free behaviour features on top of network features drops pooled R2 to 0.036-0.044, the usual too-many-features-for-~600-rows problem.
6. **Alone, the cosponsorship features predict nothing** (network [idf] only: R2 -0.001; behaviour only: -0.009).